# Mock Interview Coach - local Conversational LLM

This notebook covers the logic and prompts behind Agent 10 (Mock Interview Coach). The model acts as a technical or behavioral interviewer, evaluating the candidate's responses using the STAR method and outputting a tailored follow-up question.

In [1]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Prompt Formatting & Conversation Context

We specify the target interviewer persona (e.g. encouraging mentor vs. critical roast reviewer) and feed the running chat transcript to the model.

In [2]:
persona = "critical roast reviewer" # Options: professional, encouraging, critical roast reviewer

personas = {
    "professional": "Conduct the interview in a formal, direct, and business-appropriate style.",
    "encouraging": "Be a supportive, constructive mentor. Highlight strengths and suggest improvements gently.",
    "critical roast reviewer": "Be an extremely tough, critical reviewer. Point out logical flaws, weak metrics, and resume inflation directly and sarcastically."
}

system_prompt = f"""
You are an expert technical interviewer. Persona: {personas[persona]}
Evaluate the candidate's latest response. Provide critical feedback on content and delivery (e.g. alignment to the STAR framework), then ask a follow-up question.
"""

jd = "Senior Backend Engineer (Python, FastAPI, scaling distributed backends)"
candidate_profile = "Alice, built VinoMetrix prediction backend reducing latencies by 40% using FastAPI."
history = """
Interviewer: Welcome, Alice. Can you describe a challenging technical problem you solved on VinoMetrix?
Candidate: Yes, our prediction API was too slow under load, taking 800ms. I rewrote the prediction loops using numpy and optimized database queries, bringing response times down to 100ms.
"""

user_input = f"""
Target Job: {jd}
Candidate Profile: {candidate_profile}
Chat History:
{history}
"""

prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_input}\n<|assistant|>\n"

## 2. Load Model and Generate Response

In [3]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=350, temperature=0.7, do_sample=True)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("\n--- Interviewer Feedback & Question ---\n")
    print(result.replace(prompt, "").strip())
except Exception as e:
    print(f"Skipping model load during demo. Details: {e}")

Skipping model load during demo. Details: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
